# 02. Tiền xử lý Dữ liệu, Kỹ thuật Đặc trưng (Feature Engineering) & Chia tập
**Đề tài:** Dự đoán khả năng khách hàng hủy đặt phòng khách sạn (*Hotel Booking Demand*)

## Nguyên tắc chống rò rỉ dữ liệu (Anti-Leakage Protocol)
- Notebook này **chỉ thực hiện các phép biến đổi độc lập từng dòng (row-level)** và khử trùng lặp/bất thường cơ học.
- Mọi phép biến đổi học tham số từ phân phối dữ liệu (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `TargetEncoder`) đều được đóng gói trong `scikit-learn Pipeline` và chỉ `fit` bên trong từng fold huấn luyện ở Notebook 03.
- Đầu ra lưu dưới dạng **`.csv`** và **`.json`** trong `data/processed/` gồm hai bộ chia dữ liệu:
  1. **Stratified Holdout (80/20)**: `train_strat.csv`, `test_strat.csv` (tập chính).
  2. **Out-of-Time Split (OOT)**: `train_oot.csv` (`07/2015–04/2017`), `test_oot.csv` (`05/2017–08/2017`) phục vụ kiểm thử độ ổn định theo thời gian.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
RAW_PATH = Path("../data/raw/hotel_bookings.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH)
print(f"Dữ liệu thô ban đầu: {df_raw.shape}")

Dữ liệu thô ban đầu: (119390, 32)


## 1. Loại bỏ cột Leakage, Khử trùng lặp & Làm sạch bản ghi bất hợp lệ

In [2]:
df = df_raw.copy()

# Step 1: Drop direct target leakage columns
leakage_cols = ["reservation_status", "reservation_status_date"]
df = df.drop(columns=leakage_cols)
print(f"Sau khi loại bỏ leakage ({leakage_cols}): {df.shape}")

# Step 2: Deduplicate exact rows to prevent identical rows leaking between train and test
before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Sau khi khử trùng lặp: {df.shape} (đã loại {before_dedup - len(df):,} dòng trùng)")

# Step 3: Remove logically invalid records & extreme ADR outlier
df["children"] = df["children"].fillna(0).astype(int)
invalid_guests_mask = (df["adults"] + df["children"] + df["babies"]) == 0
neg_adr_mask = df["adr"] < 0
extreme_adr_mask = df["adr"] > 1000

print(f"Loại bỏ: 0 khách = {invalid_guests_mask.sum()}, ADR < 0 = {neg_adr_mask.sum()}, ADR > 1000 = {extreme_adr_mask.sum()}")
df = df[~(invalid_guests_mask | neg_adr_mask | extreme_adr_mask)].reset_index(drop=True)
print(f"Kích thước sau làm sạch: {df.shape}")
print("Tỷ lệ biến mục tiêu is_canceled sau làm sạch:")
print(df["is_canceled"].value_counts(normalize=True).round(4).to_dict())

Sau khi loại bỏ leakage (['reservation_status', 'reservation_status_date']): (119390, 30)
Sau khi khử trùng lặp: (87138, 30) (đã loại 32,252 dòng trùng)
Loại bỏ: 0 khách = 166, ADR < 0 = 1, ADR > 1000 = 1
Kích thước sau làm sạch: (86970, 30)
Tỷ lệ biến mục tiêu is_canceled sau làm sạch:
{0: 0.7269, 1: 0.2731}


## 2. Xử lý dữ liệu thiếu theo ngữ cảnh nghiệp vụ (Domain-Aware Missing Handling)
- `country`: điền `'Unknown'`.
- `agent`: tạo cờ nhị phân `has_agent`, chuẩn hóa ID về chuỗi sạch (ví dụ `'240'`) và điền thiếu bằng `'No_Agent'`.
- `company`: tạo cờ nhị phân `has_company`, chuẩn hóa ID về chuỗi sạch và điền thiếu bằng `'No_Company'`.
- `meal`: gộp `'Undefined'` vào `'SC'` (Self Catering — cùng nghĩa không kèm bữa ăn theo chuẩn dữ liệu khách sạn Bồ Đào Nha).

In [3]:
# Missing value handling with explicit business indicators
df["country"] = df["country"].fillna("Unknown").astype(str)

df["has_agent"] = df["agent"].notna().astype(int)
df["agent"] = df["agent"].apply(lambda x: str(int(x)) if pd.notna(x) else "No_Agent")

df["has_company"] = df["company"].notna().astype(int)
df["company"] = df["company"].apply(lambda x: str(int(x)) if pd.notna(x) else "No_Company")

# Standardize meal category ('Undefined' == 'SC' in Portuguese hotel PMS)
df["meal"] = df["meal"].replace("Undefined", "SC").astype(str)

# Clean remaining string columns explicitly (Pandas 3.x safe)
str_cols = ["hotel", "market_segment", "distribution_channel", "reserved_room_type",
            "assigned_room_type", "deposit_type", "customer_type"]
for col in str_cols:
    df[col] = df[col].fillna("Unknown").astype(str)

print("Số giá trị thiếu còn lại trên toàn bộ bảng:", int(df.isna().sum().sum()))

Số giá trị thiếu còn lại trên toàn bộ bảng: 0


## 3. Kỹ thuật tạo đặc trưng mới (Feature Engineering)
Tạo nhóm 15 đặc trưng nghiệp vụ mới hoàn toàn ở mức dòng (zero leakage):
- **Cấu trúc lưu trú & khách**: `total_nights`, `total_guests`, `has_children_or_babies`, `is_zero_nights`, `weekend_ratio`.
- **Trạng thái phòng & lịch sử khách hàng**: `room_type_changed`, `total_previous_bookings`, `prev_cancel_ratio`.
- **Giá trị đơn đặt phòng**: `adr_per_person`, `estimated_total_revenue`.
- **Mức độ cam kết**: `has_special_requests`, `has_parking`.
- **Tính chu kỳ mùa vụ**: `arrival_month_num`, `arrival_month_sin`, `arrival_month_cos`.

In [4]:
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}

# Stay & guest structure
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["has_children_or_babies"] = ((df["children"] + df["babies"]) > 0).astype(int)
df["is_zero_nights"] = (df["total_nights"] == 0).astype(int)
df["weekend_ratio"] = (df["stays_in_weekend_nights"] / (df["total_nights"] + 1e-5)).round(4)

# Room & history features
df["room_type_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)
df["total_previous_bookings"] = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
df["prev_cancel_ratio"] = (df["previous_cancellations"] / (df["total_previous_bookings"] + 1e-5)).round(4)

# Monetary & commitment features
df["adr_per_person"] = (df["adr"] / df["total_guests"].clip(lower=1)).round(4)
df["estimated_total_revenue"] = (df["adr"] * df["total_nights"]).round(2)
df["has_special_requests"] = (df["total_of_special_requests"] > 0).astype(int)
df["has_parking"] = (df["required_car_parking_spaces"] > 0).astype(int)

# Cyclical month encoding + helper arrival_date for Out-of-Time split
df["arrival_month_num"] = df["arrival_date_month"].map(month_map).astype(int)
df["arrival_month_sin"] = np.sin(2 * np.pi * df["arrival_month_num"] / 12).round(6)
df["arrival_month_cos"] = np.cos(2 * np.pi * df["arrival_month_num"] / 12).round(6)

arrival_date = pd.to_datetime(
    dict(
        year=df["arrival_date_year"],
        month=df["arrival_month_num"],
        day=df["arrival_date_day_of_month"],
    )
)
print(f"Khoảng thời gian dữ liệu: {arrival_date.min().date()} -> {arrival_date.max().date()}")

# Drop redundant raw text month column (represented by arrival_month_num + sin/cos)
df = df.drop(columns=["arrival_date_month"])
print(f"Tổng số cột sau Feature Engineering (gồm target): {df.shape[1]}")

Khoảng thời gian dữ liệu: 2015-07-01 -> 2017-08-31
Tổng số cột sau Feature Engineering (gồm target): 46


## 4. Phân nhóm đặc trưng & Chia tập dữ liệu kép (Stratified 80/20 & Out-of-Time)
- **Tập chính (`train_strat.csv` / `test_strat.csv`)**: Chia ngẫu nhiên phân tầng tỷ lệ 80/20 với `RANDOM_STATE = 42`.
- **Tập đối chiếu theo thời gian (`train_oot.csv` / `test_oot.csv`)**: Huấn luyện trên quá khứ (`<= 2017-04-30`), kiểm thử trên giai đoạn mùa cao điểm tương lai (`>= 2017-05-01`).

In [5]:
target_col = "is_canceled"
high_card_cat_cols = ["country", "agent", "company"]
low_card_cat_cols = [
    "hotel", "meal", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type",
]
num_cols = [c for c in df.columns if c not in [target_col] + high_card_cat_cols + low_card_cat_cols]

print(f"Số biến số (num_cols): {len(num_cols)}")
print(f"Số biến phân loại ít giá trị (low_card_cat_cols): {len(low_card_cat_cols)}")
print(f"Số biến phân loại nhiều giá trị (high_card_cat_cols): {len(high_card_cat_cols)}")

# 4.1 Stratified 80/20 Holdout Split
train_strat, test_strat = train_test_split(
    df,
    test_size=0.20,
    stratify=df[target_col],
    random_state=RANDOM_STATE,
)
train_strat = train_strat.reset_index(drop=True)
test_strat = test_strat.reset_index(drop=True)

# 4.2 Out-of-Time (OOT) Split: Train <= 2017-04-30, Test >= 2017-05-01
oot_cutoff = pd.Timestamp("2017-05-01")
train_oot = df[arrival_date < oot_cutoff].reset_index(drop=True)
test_oot = df[arrival_date >= oot_cutoff].reset_index(drop=True)

# Integrity checks
assert df.isna().sum().sum() == 0, "Còn giá trị NaN trong dữ liệu!"
assert "reservation_status" not in df.columns and "reservation_status_date" not in df.columns
# Verify zero exact-row overlap between train_strat and test_strat
train_hashes = set(pd.util.hash_pandas_object(train_strat, index=False))
test_hashes = set(pd.util.hash_pandas_object(test_strat, index=False))
overlap_count = len(train_hashes.intersection(test_hashes))
assert overlap_count == 0, f"Phát hiện {overlap_count} bản ghi trùng giữa train và test!"
assert arrival_date[arrival_date < oot_cutoff].max() < arrival_date[arrival_date >= oot_cutoff].min()

# Save to CSV & JSON
train_strat.to_csv(PROCESSED_DIR / "train_strat.csv", index=False)
test_strat.to_csv(PROCESSED_DIR / "test_strat.csv", index=False)
train_oot.to_csv(PROCESSED_DIR / "train_oot.csv", index=False)
test_oot.to_csv(PROCESSED_DIR / "test_oot.csv", index=False)

metadata = {
    "random_state": RANDOM_STATE,
    "target_col": target_col,
    "num_cols": num_cols,
    "low_card_cat_cols": low_card_cat_cols,
    "high_card_cat_cols": high_card_cat_cols,
    "oot_cutoff_date": str(oot_cutoff.date()),
    "shapes": {
        "cleaned_total": list(df.shape),
        "train_strat": list(train_strat.shape),
        "test_strat": list(test_strat.shape),
        "train_oot": list(train_oot.shape),
        "test_oot": list(test_oot.shape),
    },
    "target_rates": {
        "cleaned_total": round(float(df[target_col].mean()), 4),
        "train_strat": round(float(train_strat[target_col].mean()), 4),
        "test_strat": round(float(test_strat[target_col].mean()), 4),
        "train_oot": round(float(train_oot[target_col].mean()), 4),
        "test_oot": round(float(test_oot[target_col].mean()), 4),
    },
    "train_test_exact_overlap": overlap_count,
}
with (PROCESSED_DIR / "feature_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

summary_splits = pd.DataFrame([
    {"Tập dữ liệu": "train_strat.csv (80%)", "Số dòng": len(train_strat), "Số cột": train_strat.shape[1], "Tỷ lệ hủy (%)": round(train_strat[target_col].mean()*100, 2)},
    {"Tập dữ liệu": "test_strat.csv (20%)", "Số dòng": len(test_strat), "Số cột": test_strat.shape[1], "Tỷ lệ hủy (%)": round(test_strat[target_col].mean()*100, 2)},
    {"Tập dữ liệu": "train_oot.csv (<= 04/2017)", "Số dòng": len(train_oot), "Số cột": train_oot.shape[1], "Tỷ lệ hủy (%)": round(train_oot[target_col].mean()*100, 2)},
    {"Tập dữ liệu": "test_oot.csv (>= 05/2017)", "Số dòng": len(test_oot), "Số cột": test_oot.shape[1], "Tỷ lệ hủy (%)": round(test_oot[target_col].mean()*100, 2)},
])
display(summary_splits)
print("Kiểm tra toàn vẹn thành công: 0 NaN, 0 rò rỉ nhãn, 0 dòng trùng lặp giữa Train/Test!")

Số biến số (num_cols): 34
Số biến phân loại ít giá trị (low_card_cat_cols): 8
Số biến phân loại nhiều giá trị (high_card_cat_cols): 3
Kiểm tra toàn vẹn thành công: 0 NaN, 0 rò rỉ nhãn, 0 dòng trùng lặp giữa Train/Test!


,Tập dữ liệu,Số dòng,Số cột,Tỷ lệ hủy (%)
0,train_strat.csv (80%),69576,46,27.31
1,test_strat.csv (20%),17394,46,27.31
2,train_oot.csv (<= 04/2017),69295,46,25.46
3,test_oot.csv (>= 05/2017),17675,46,34.55
